# Etapa Gold: métricas para la Intendencia de Montevideo

Cada métrica se publica como una tabla Delta en `movilidad_gold`, pensada para alimentar directamente un gráfico del dashboard.

| Bloque | Tabla gold | Pregunta que responde |
|---|---|---|
| 1 | `viajes_mes`, `estacionalidad_mes` | ¿Cuál es el mes más activo? ¿Cómo evolucionan los viajes? |
| 2 | `demanda_tipo_dia`, `demanda_tipo_dia_hora`, `demanda_vs_oferta_tipo_dia_hora` | ¿Cómo cambia la demanda el fin de semana y baja la oferta lo mismo? |
| 3 | `pago_hora`, `pago_mes` | ¿Cómo se paga según la hora (tarjeta o efectivo)? |
| 4 | `linea_mes`, `linea_tendencia`, `linea_concentracion_pico` | ¿Qué líneas crecen o caen? ¿Cuáles son pendulares? |
| 5 | `paradas_compartidas`, `paradas_lineas_distribucion`, `oferta_parada_hora`, `huecos_servicio` | Nodos de transbordo, frecuencia y huecos de servicio |
| 6 | `demanda_oferta_parada_hora` | Paradas con demanda alta y poca oferta (o al revés) |
| 7 | `cobertura_hex_hora`, `desiertos_nocturnos`, `demanda_oferta_hex_hora` | Mapa de cobertura por zona y hora (requiere `paradas_ubicacion` en silver) |
| 8 | `calidad_viajes` | ¿Cuántos registros se excluyen y por qué? (sobre una muestra) |

**Decisiones generales**
- Demanda = `SUM(cantidad_pasajeros)`, un registro de silver es un tramo y puede llevar varios pasajeros.
- Se excluyen solo las filas con `CON_TARJETA_INVALIDA_O_AUSENTE`, `CANTIDAD_PASAJEROS_INVALIDA` o `FECHA_EVENTO_AUSENTE`, las demás marcas de silver no afectan a estas métricas, porque se agrupa por `fecha_evento`. El bloque final (`calidad_viajes`) lo cuantifica sobre una muestra.
- `tipo_dia` (HABIL, SABADO, DOMINGO) sale del día de la semana del **día de servicio** (04:00 a 04:00), igual que los horarios. Un viaje a la 01:30 del sábado cuenta como día hábil si el viernes fue hábil. **No se tratan feriados** como domingo.
- Un mes cuenta como completo solo si tiene datos en todos sus días, las comparaciones entre meses usan únicamente meses completos.
- Los horarios (`horarios_por_parada`) son una foto de la estipulado, sin fecha de vigencia, se compara contra la demanda reciente, no contra toda la historia.

In [ ]:
from datetime import date
from pyspark.sql import functions as F, Window as W

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema_silver", "movilidad_silver")
dbutils.widgets.text("schema_gold", "movilidad_gold")
dbutils.widgets.text("schema_trabajo", "movilidad_trabajo")
# Ventana de demanda reciente (en meses completos) para comparar contra la oferta actual
dbutils.widgets.text("meses_demanda", "6")
# Ventana para calcular tendencias de líneas (en meses completos)
dbutils.widgets.text("meses_tendencia", "24")
# Share mínimo promedio (en %) para que una línea entre al ranking de tendencias
dbutils.widgets.text("share_minimo_pct", "0.05")
# Si horarios_por_parada u horarios_circulares tienen más de un archivo de origen, elegir cuál usar
dbutils.widgets.text("archivo_horarios", "")
dbutils.widgets.text("archivo_circulares", "")
# Fracción de viajes_stm usada para el resumen de calidad (1 = exacto, recorre toda la tabla y tarda)
dbutils.widgets.text("calidad_muestra", "0.01")
# Incluir también las pasadas de líneas circulares (horarios_circulares) en la oferta: si / no
dbutils.widgets.text("incluir_circulares", "si")

CATALOG = dbutils.widgets.get("catalog")
SILVER = dbutils.widgets.get("schema_silver")
GOLD = dbutils.widgets.get("schema_gold")
TRABAJO = dbutils.widgets.get("schema_trabajo")
MESES_DEMANDA = int(dbutils.widgets.get("meses_demanda"))
MESES_TENDENCIA = int(dbutils.widgets.get("meses_tendencia"))
SHARE_MINIMO = float(dbutils.widgets.get("share_minimo_pct"))
CALIDAD_MUESTRA = float(dbutils.widgets.get("calidad_muestra"))
ARCHIVO_HORARIOS = dbutils.widgets.get("archivo_horarios").strip()
ARCHIVO_CIRCULARES = dbutils.widgets.get("archivo_circulares").strip()
INCLUIR_CIRCULARES = dbutils.widgets.get("incluir_circulares").strip().lower() == "si"

# El día de servicio va de 04:00 a 04:00 (según la documentación de los horarios)
INICIO_DIA_SERVICIO_H = 4

spark.conf.set("spark.sql.session.timeZone", "America/Montevideo")

for esquema in (GOLD, TRABAJO):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{esquema}")


def leer(esquema, tabla):
    return spark.table(f"{CATALOG}.{esquema}.{tabla}")


def guardar(df, tabla, esquema=None):
    """Publica df como tabla Delta (overwrite) y devuelve la tabla leída desde disco."""
    esquema = esquema or GOLD
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{esquema}.{tabla}")
    )
    print("Tabla publicada:", f"{CATALOG}.{esquema}.{tabla}")
    return leer(esquema, tabla)


# Reglas de silver que excluyen una fila de las métricas de demanda
REGLAS_EXCLUYENTES = [
    "CON_TARJETA_INVALIDA_O_AUSENTE",
    "CANTIDAD_PASAJEROS_INVALIDA",
    "FECHA_EVENTO_AUSENTE",
]
REGLAS_SILVER = REGLAS_EXCLUYENTES + [
    "PERIODO_FUENTE_INVALIDO",
    "MES_DISTINTO_DE_PERIODO_FUENTE",
    "ORDINAL_INVALIDO_CON_TARJETA",
    "ORDINAL_NEGATIVO_SIN_TARJETA",
]

# True si la fila NO tiene ninguna regla excluyente (un array nulo se considera sin problemas)
fila_valida = ~F.coalesce(
    F.arrays_overlap(
        F.col("problemas_validacion"),
        F.array(*[F.lit(r) for r in REGLAS_EXCLUYENTES]),
    ),
    F.lit(False),
)


def con_calendario(df):
    """Agrega fecha, fecha_servicio, hora, anio, mes y tipo_dia a partir de fecha_evento.

    fecha, anio y mes son de calendario. tipo_dia se calcula sobre el día de servicio
    (04:00 a 04:00), igual que en los horarios: un viaje a la 01:30 del sábado pertenece
    al día de servicio del viernes (HABIL).
    """
    servicio = F.col("fecha_evento") - F.expr(f"INTERVAL {INICIO_DIA_SERVICIO_H} HOURS")
    return (
        df.withColumn("fecha", F.to_date("fecha_evento"))
        .withColumn("fecha_servicio", F.to_date(servicio))
        .withColumn("hora", F.hour("fecha_evento"))
        .withColumn("anio", F.year("fecha_evento"))
        .withColumn("mes", F.month("fecha_evento"))
        .withColumn(
            "tipo_dia",
            F.when(F.dayofweek("fecha_servicio") == 1, "DOMINGO")
            .when(F.dayofweek("fecha_servicio") == 7, "SABADO")
            .otherwise("HABIL"),
        )
    )


def sin_problemas(columna="problemas_validacion"):
    return F.coalesce(F.size(columna), F.lit(0)) <= 0

## Tabla de trabajo: viajes por día, hora, forma de pago y línea

Se agrega una sola vez el volumen completo de viajes a la granularidad (fecha, hora, forma de pago, línea) y se materializa en `movilidad_trabajo.viajes_dia_hora_linea`. Los bloques 1 a 4 se calculan desde esta tabla, mucho más chica que `viajes_stm`.

Se hace un solo recorrido y se agrupa por fecha, hora, forma de pago y línea, sumando pasajeros y tramos: cada fila de la tabla resultante resume miles de viajes.

`forma_pago`: `con_tarjeta = 1` es TARJETA y `con_tarjeta = 0` es EFECTIVO.

In [ ]:
viajes_silver = leer(SILVER, "viajes_stm")

viajes_agg = (
    con_calendario(viajes_silver.filter(fila_valida))
    .withColumn("forma_pago", F.when(F.col("con_tarjeta") == 1, "TARJETA").otherwise("EFECTIVO"))
    .groupBy("fecha", "fecha_servicio", "anio", "mes", "tipo_dia", "hora", "forma_pago", "linea_codigo", "dsc_linea")
    .agg(
        F.count("*").alias("tramos"),
        F.sum("cantidad_pasajeros").alias("pasajeros"),
    )
)
viajes_agg = guardar(viajes_agg, "viajes_dia_hora_linea", TRABAJO)
display(viajes_agg.limit(10))

## 1. Evolución de viajes y mes más activo

`viajes_mes`: pasajeros, tramos y **pasajeros por día** de cada mes. Normalizar por día evita que febrero "pierda" siempre contra los meses de 31 días. `mes_completo` marca los meses con datos en todos sus días.

`estacionalidad_mes`: para quitar el efecto del crecimiento entre años, se divide el promedio diario de cada mes por el promedio de su año. Solo se usan años con los 12 meses completos (2026 excluido). Un índice de 1,10 significa un 10 % por encima de un mes promedio de ese año.

In [ ]:
viajes_mes = (
    viajes_agg.groupBy("anio", "mes")
    .agg(
        F.sum("pasajeros").alias("pasajeros"),
        F.sum("tramos").alias("tramos"),
        F.countDistinct("fecha").alias("dias_con_datos"),
    )
    .withColumn("dias_calendario", F.dayofmonth(F.last_day(F.make_date("anio", "mes", F.lit(1)))))
    .withColumn("pasajeros_por_dia", F.col("pasajeros") / F.col("dias_con_datos"))
    .withColumn("mes_completo", F.col("dias_con_datos") == F.col("dias_calendario"))
)
viajes_mes = guardar(viajes_mes, "viajes_mes")
display(viajes_mes.orderBy("anio", "mes"))

In [ ]:
anios_completos = (
    viajes_mes.filter("mes_completo")
    .groupBy("anio")
    .agg(F.count("*").alias("meses_completos"))
    .filter(F.col("meses_completos") == 12)
    .select("anio")
)

estacionalidad_mes = (
    viajes_mes.filter("mes_completo")
    .join(anios_completos, "anio")
    .withColumn("indice_estacional", F.col("pasajeros_por_dia") / F.avg("pasajeros_por_dia").over(W.partitionBy("anio")))
    .groupBy("mes")
    .agg(
        F.avg("indice_estacional").alias("indice_estacional_prom"),
        F.count("*").alias("anios_considerados"),
    )
    .withColumn("ranking", F.rank().over(W.orderBy(F.desc("indice_estacional_prom"))))
)
estacionalidad_mes = guardar(estacionalidad_mes, "estacionalidad_mes")
display(estacionalidad_mes.orderBy("mes"))

## 2. Fin de semana v.s. semana y demanda v.s. oferta

`demanda_tipo_dia_hora`: responde ¿a qué hora viaja la gente, según sea día hábil, sábado o domingo?, para cada año, tipo de día y hora, calcula cuántos pasajeros suben en promedio, se divide por la cantidad de días de ese tipo (columna dias) para que sea un promedio por día y se puedan comparar tipos de día con distinta cantidad de días, además sirve para detectar años incompletos: si 2026 solo tiene 10 domingos, ese año tiene menos datos.

`demanda_tipo_dia`: pasajeros por día de cada tipo y su proporción respecto de un día hábil, responde a: cuando baja la demanda el fin de semana, ¿baja también el servicio en la misma proporción?

`demanda_vs_oferta_tipo_dia_hora`: compara la **caída de la demanda** con la **caída de la oferta programada** de sábado y domingo respecto de un día hábil. La oferta son las pasadas programadas por hora en `horarios_por_parada` (indicador proporcional al servicio). La demanda es la de los últimos `meses_demanda` meses completos.

Por ende, si `brecha_oferta_demanda` es:
- Negativa: el servicio bajó más que la gente, hay menos ómnibus de los que se esperarían para esa demanda, y puede ser un problema de servicio.
- Cerca de cero: el servicio acompañó a la demanda.
- Positiva: el servicio bajó menos que la demanda, o sea hay más ómnibus de los que se necesitan.

In [ ]:
dias_tipo = viajes_agg.groupBy("anio", "tipo_dia").agg(F.countDistinct("fecha_servicio").alias("dias"))

demanda_tipo_dia_hora = (
    viajes_agg.groupBy("anio", "tipo_dia", "hora")
    .agg(F.sum("pasajeros").alias("pasajeros"))
    .join(dias_tipo, ["anio", "tipo_dia"])
    .withColumn("pasajeros_por_dia", F.col("pasajeros") / F.col("dias"))
)
demanda_tipo_dia_hora = guardar(demanda_tipo_dia_hora, "demanda_tipo_dia_hora")

demanda_tipo_dia = (
    viajes_agg.groupBy("anio", "tipo_dia")
    .agg(F.sum("pasajeros").alias("pasajeros"))
    .join(dias_tipo, ["anio", "tipo_dia"])
    .withColumn("pasajeros_por_dia", F.col("pasajeros") / F.col("dias"))
    .withColumn(
        "proporcion_vs_habil",
        F.col("pasajeros_por_dia")
        / F.max(F.when(F.col("tipo_dia") == "HABIL", F.col("pasajeros_por_dia"))).over(W.partitionBy("anio")),
    )
)
demanda_tipo_dia = guardar(demanda_tipo_dia, "demanda_tipo_dia")
display(demanda_tipo_dia.orderBy("anio", "tipo_dia"))

### Ventana de demanda reciente

Los últimos `meses_demanda` meses **completos**: el mes en curso, si está incompleto, queda afuera.

In [ ]:
ultimo = (
    viajes_mes.filter("mes_completo")
    .orderBy(F.desc("anio"), F.desc("mes"))
    .first()
)
assert ultimo is not None, "No hay ningún mes completo en viajes_mes"

# hasta es exclusivo: primer día del mes siguiente al último mes completo
VENTANA_HASTA = spark.sql(f"SELECT add_months(make_date({ultimo['anio']}, {ultimo['mes']}, 1), 1)").first()[0]
VENTANA_DESDE = spark.sql(f"SELECT add_months(DATE'{VENTANA_HASTA}', -{MESES_DEMANDA})").first()[0]
print(f"Ventana de demanda reciente: {VENTANA_DESDE} <= fecha < {VENTANA_HASTA}")

en_ventana = (F.col("fecha") >= F.lit(VENTANA_DESDE)) & (F.col("fecha") < F.lit(VENTANA_HASTA))

dias_ventana = (
    viajes_agg.filter(en_ventana)
    .groupBy("tipo_dia")
    .agg(F.countDistinct("fecha_servicio").alias("dias"))
)
display(dias_ventana)

### Oferta programada (horarios por parada y circulares)

Se usan las pasadas programadas de `horarios_por_parada` y, si `incluir_circulares = si`, también las de `horarios_circulares` (líneas circulares). Solo se excluyen registros con una marca que afecta al cálculo:
- paradas: `TIPO_DIA_INVALIDO_O_AUSENTE`, `HORA_PARA_REVISAR`
- circulares: `TIPO_DIA_AUSENTE_O_NO_INTERPRETABLE`, `TIPO_DIA_NO_DOCUMENTADO`, `HORA_PARA_REVISAR`
- y errores de conversión en `tipo_dia`, `cod_variante`, `cod_ubic_parada` u `hora` - (en silver fallaron al convertir a número algún campo clave)

Las marcas `ORDINAL_PARA_REVISAR`, `DIA_ANTERIOR_INVALIDO_O_AUSENTE` y `PARTE_PARA_REVISAR` **no** excluyen: no cambian a qué parada, tipo de día y hora corresponde la pasada.

Se eliminan copias exactas de la misma pasada (variante, frecuencia, parada, ordinal, tipo de día, hora y `dia_anterior`), lo que también evita contar dos veces una pasada que esté en ambas tablas.

**Si una tabla tiene más de un archivo de origen, hay que elegir cuál usar** (widgets `archivo_horarios` y `archivo_circulares`). De lo contrario, las pasadas se contarían varias veces. Esto es porque cada tabla guarda de qué archivo vino cada fila (archivo_origen), si se cargó varias versiones de los horarios, por ejemplo de distintos meses, la misma pasada aparecería una vez por archivo y el servicio se contaría doble o triple. Por eso el notebook se detiene y pide elegir uno en los widgets, si hay un solo archivo, sigue solo.

In [ ]:
COLUMNAS_PASADA = [
    "cod_variante", "frecuencia", "cod_ubic_parada", "ordinal",
    "tipo_dia_normalizado", "hora", "dia_anterior", "segundos_desde_medianoche",
]
CONVERSION_CRITICA = ["tipo_dia", "cod_variante", "cod_ubic_parada", "hora"]


def sin_reglas(reglas, columna="problemas_validacion"):
    """True si la fila no tiene ninguna de las reglas indicadas (array nulo = sin problemas)."""
    return ~F.coalesce(
        F.arrays_overlap(F.col(columna), F.array(*[F.lit(r) for r in reglas])), F.lit(False)
    )


def elegir_archivo(df, nombre, elegido):
    archivos = [r[0] for r in df.select("archivo_origen").distinct().collect()]
    print(f"Archivos de origen en {nombre}:", archivos)
    if elegido:
        return df.filter(F.col("archivo_origen") == elegido)
    if len(archivos) > 1:
        raise ValueError(
            f"{nombre} tiene más de un archivo de origen. Completar el widget correspondiente con uno de: {archivos}"
        )
    return df


def pasadas_validas(df, reglas_excluyentes):
    return df.filter(
        sin_reglas(reglas_excluyentes)
        & sin_reglas(CONVERSION_CRITICA, "errores_conversion")
        & F.col("segundos_desde_medianoche").isNotNull()
        & F.col("tipo_dia_normalizado").isin("HABIL", "SABADO", "DOMINGO")
        & F.col("cod_ubic_parada").isNotNull()
        & F.col("cod_variante").isNotNull()
    )


horarios_raw = elegir_archivo(leer(SILVER, "horarios_por_parada"), "horarios_por_parada", ARCHIVO_HORARIOS)
pasadas_var = pasadas_validas(horarios_raw, ["TIPO_DIA_INVALIDO_O_AUSENTE", "HORA_PARA_REVISAR"]).select(*COLUMNAS_PASADA)

resumen = {
    "variantes_entrada": horarios_raw.count(),
    "variantes_validas": pasadas_var.count(),
}

if INCLUIR_CIRCULARES:
    circ_raw = elegir_archivo(leer(SILVER, "horarios_circulares"), "horarios_circulares", ARCHIVO_CIRCULARES)
    pasadas_circ = pasadas_validas(
        circ_raw,
        ["TIPO_DIA_AUSENTE_O_NO_INTERPRETABLE", "TIPO_DIA_NO_DOCUMENTADO", "HORA_PARA_REVISAR"],
    ).select(*COLUMNAS_PASADA)
    resumen["circulares_entrada"] = circ_raw.count()
    resumen["circulares_validas"] = pasadas_circ.count()
    pasadas = pasadas_var.unionByName(pasadas_circ)
else:
    pasadas = pasadas_var

horarios = pasadas.dropDuplicates(COLUMNAS_PASADA).withColumn(
    "hora_h", F.floor(F.col("segundos_desde_medianoche") / 3600).cast("int")
)
resumen["pasadas_sin_copias"] = horarios.count()

display(spark.createDataFrame([tuple(resumen.values())], ", ".join(f"{k} long" for k in resumen)))

In [ ]:
oferta_parada_hora = (
    horarios.groupBy("cod_ubic_parada", "tipo_dia_normalizado", "hora_h")
    .agg(
        F.count("*").alias("salidas"),
        F.countDistinct("cod_variante").alias("variantes"),
    )
    .withColumnRenamed("cod_ubic_parada", "codigo_parada")
    .withColumnRenamed("tipo_dia_normalizado", "tipo_dia")
    .withColumnRenamed("hora_h", "hora")
    .withColumn("headway_min", F.lit(60.0) / F.col("salidas"))
)
oferta_parada_hora = guardar(oferta_parada_hora, "oferta_parada_hora")
display(oferta_parada_hora.limit(10))

Headway es el termino para describir el tiempo promedio de espera entre un ómnibus y el siguiente, en minutos, en una parada y una hora determinada.

`headway_min = 60 / salidas` asume pasadas parejas dentro de la hora: es un promedio, no el intervalo real entre ómnibus.

_Cuanto menor es el headway, mejor el servicio, porque se espera menos_

**Dos aclaraciones:**
1. Es un promedio, supone que las pasadas están repartidas parejo dentro de la hora; si pasan 4 ómnibus juntos en los primeros 10 minutos, el cálculo dice 15 minutos igual, aunque en la práctica se espere mucho más sirve para comparar paradas y horas entre sí, no para decir cuánto espera una persona.
2. Cuenta todas las líneas de la parada, si pasan dos líneas distintas, sus pasadas se suman. Es el tiempo de espera hasta el próximo ómnibus de cualquiera de ellas, no de una línea en particular.

In [ ]:
oferta_tipo_dia_hora = (
    oferta_parada_hora.groupBy("tipo_dia", "hora")
    .agg(F.sum("salidas").alias("pasadas_programadas"))
)

demanda_reciente = (
    viajes_agg.filter(en_ventana)
    .groupBy("tipo_dia", "hora")
    .agg(F.sum("pasajeros").alias("pasajeros"))
    .join(dias_ventana, "tipo_dia")
    .withColumn("pasajeros_por_dia", F.col("pasajeros") / F.col("dias"))
)

w_habil = W.partitionBy("hora")


def indice_vs_habil(columna):
    habil = F.max(F.when(F.col("tipo_dia") == "HABIL", F.col(columna))).over(w_habil)
    return F.col(columna) / habil


demanda_vs_oferta = (
    demanda_reciente.join(oferta_tipo_dia_hora, ["tipo_dia", "hora"], "full")
    .fillna(0, subset=["pasajeros_por_dia", "pasadas_programadas"])
    .withColumn("demanda_vs_habil", indice_vs_habil("pasajeros_por_dia"))
    .withColumn("oferta_vs_habil", indice_vs_habil("pasadas_programadas"))
    .withColumn("brecha_oferta_demanda", F.col("oferta_vs_habil") - F.col("demanda_vs_habil"))
)
demanda_vs_oferta = guardar(demanda_vs_oferta, "demanda_vs_oferta_tipo_dia_hora")
display(demanda_vs_oferta.orderBy("tipo_dia", "hora"))

**Recordatorio**

`brecha_oferta_demanda` es:
- Negativa: el servicio bajó más que la gente, hay menos ómnibus de los que se esperarían para esa demanda, y puede ser un problema de servicio.
- Cerca de cero: el servicio acompañó a la demanda.
- Positiva: el servicio bajó menos que la demanda, o sea hay más ómnibus de los que se necesitan.

## 3. Forma de pago según la hora

`pago_hora`: pasajeros y porcentaje por forma de pago, hora y tipo de día. 

`pago_mes`: evolución mensual del porcentaje de efectivo.

In [ ]:
pago_hora = (
    viajes_agg.groupBy("anio", "tipo_dia", "hora", "forma_pago")
    .agg(F.sum("pasajeros").alias("pasajeros"))
    .withColumn(
        "porcentaje",
        100 * F.col("pasajeros") / F.sum("pasajeros").over(W.partitionBy("anio", "tipo_dia", "hora")),
    )
)
pago_hora = guardar(pago_hora, "pago_hora")

pago_mes = (
    viajes_agg.groupBy("anio", "mes", "forma_pago")
    .agg(F.sum("pasajeros").alias("pasajeros"))
    .withColumn(
        "porcentaje",
        100 * F.col("pasajeros") / F.sum("pasajeros").over(W.partitionBy("anio", "mes")),
    )
)
pago_mes = guardar(pago_mes, "pago_mes")
display(pago_mes.orderBy("anio", "mes", "forma_pago"))

## 4. Líneas en alza y en baja, y concentración en hora pico

Este bloque busca responder ¿qué líneas se usan cada vez más y cuáles cada vez menos?

`linea_mes`: para cada línea y cada mes almace dos cosas

- Pasajeros por día: cuánta gente usa la línea en promedio por día
- Share: qué porcentaje del total del sistema representa esa línea ese mes (mes completo). Ejemplo: si el sistema tuvo 1.000.000 de pasajeros por día y la línea 117 tuvo 20.000, su share es 2 %.

`linea_tendencia`: mira cómo cambia el share de cada línea mes a mes dentro de una ventana (por defecto los últimos 24 meses completos) y le ajusta una recta, acá aplica lo básico de la matemática, la inclinación de esa recta es la pendiente: si sube, la línea gana peso, y si baja, lo pierde. `regr_slope` es la función de Spark que calcula esa inclinación.

Da dos números:

- `pendiente_share_pp_anio`: cuántos puntos porcentuales de share gana o pierde la línea por año

    Ejemplo: +0,5 significa que pasó, digamos, de 2,0 % a 2,5 % del sistema en un año. Compara las líneas entre sí en términos absolutos, así que las líneas grandes tienden a quedar arriba del ranking.

- `variacion_anual_pct`: cuánto cambian sus pasajeros por día por año, en relación a lo que tenía

    Ejemplo: +10 % significa que cada año tiene un 10 % más de pasajeros por día. Sirve para ver líneas chicas que crecen mucho en proporción.

El ranking de alza y de baja usa la primera (`pendiente_share_pp_anio`).

In [ ]:
meses_ok = (
    viajes_mes.filter("mes_completo")
    .select("anio", "mes", F.col("pasajeros").alias("pasajeros_total"), "dias_con_datos")
)

linea_mes = (
    viajes_agg.filter(F.col("linea_codigo").isNotNull())
    .groupBy("linea_codigo", "anio", "mes")
    .agg(
        F.sum("pasajeros").alias("pasajeros"),
        F.max_by("dsc_linea", "fecha").alias("dsc_linea"),
    )
    .join(meses_ok, ["anio", "mes"])
    .withColumn("mes_idx", F.col("anio") * 12 + F.col("mes"))
    .withColumn("pasajeros_por_dia", F.col("pasajeros") / F.col("dias_con_datos"))
    .withColumn("share_pct", 100 * F.col("pasajeros") / F.col("pasajeros_total"))
    .drop("pasajeros_total", "dias_con_datos")
)
linea_mes = guardar(linea_mes, "linea_mes")

In [ ]:
max_idx = linea_mes.agg(F.max("mes_idx")).first()[0]
MIN_MESES = int(MESES_TENDENCIA * 0.75)

linea_tendencia = (
    linea_mes.filter(F.col("mes_idx") > max_idx - MESES_TENDENCIA)
    .groupBy("linea_codigo")
    .agg(
        F.max_by("dsc_linea", "mes_idx").alias("linea"),
        F.count("*").alias("meses_con_datos"),
        F.avg("share_pct").alias("share_prom_pct"),
        F.avg("pasajeros_por_dia").alias("pasajeros_por_dia_prom"),
        (F.expr("regr_slope(share_pct, mes_idx)") * 12).alias("pendiente_share_pp_anio"),
        (F.expr("regr_slope(pasajeros_por_dia, mes_idx)") * 12).alias("pendiente_pasajeros_dia_anio"),
    )
    .filter((F.col("meses_con_datos") >= MIN_MESES) & (F.col("share_prom_pct") >= SHARE_MINIMO))
    .withColumn("variacion_anual_pct", 100 * F.col("pendiente_pasajeros_dia_anio") / F.col("pasajeros_por_dia_prom"))
    .withColumn("tendencia", F.when(F.col("pendiente_share_pp_anio") >= 0, "EN_ALZA").otherwise("EN_BAJA"))
    .withColumn("ranking_alza", F.row_number().over(W.orderBy(F.desc("pendiente_share_pp_anio"))))
    .withColumn("ranking_baja", F.row_number().over(W.orderBy(F.asc("pendiente_share_pp_anio"))))
)
linea_tendencia = guardar(linea_tendencia, "linea_tendencia")

print("Top 10 en alza")
display(linea_tendencia.orderBy("ranking_alza").limit(10))
print("Top 10 en baja")
display(linea_tendencia.orderBy("ranking_baja").limit(10))

### Concentración en hora pico por línea

Porcentaje de los pasajeros de días hábiles (en la ventana de demanda reciente) que viaja en las 3 horas más cargadas de la línea. Valores altos indican líneas **pendulares** (trabajo y estudio). Valores bajos indican uso más parejo a lo largo del día.

In [ ]:
por_linea_hora = (
    viajes_agg.filter(en_ventana & (F.col("tipo_dia") == "HABIL") & F.col("linea_codigo").isNotNull())
    .groupBy("linea_codigo", "hora")
    .agg(F.sum("pasajeros").alias("pasajeros"), F.max_by("dsc_linea", "fecha").alias("dsc_linea"))
    .withColumn("share_hora", F.col("pasajeros") / F.sum("pasajeros").over(W.partitionBy("linea_codigo")))
    .withColumn("pos", F.row_number().over(W.partitionBy("linea_codigo").orderBy(F.desc("pasajeros"))))
)

linea_concentracion_pico = (
    por_linea_hora.groupBy("linea_codigo")
    .agg(
        F.max_by("dsc_linea", "pasajeros").alias("linea"),
        F.sum("pasajeros").alias("pasajeros"),
        (100 * F.sum(F.when(F.col("pos") <= 3, F.col("share_hora")).otherwise(0))).alias("pct_en_3_horas_pico"),
        F.max(F.when(F.col("pos") == 1, F.col("hora"))).alias("hora_pico"),
    )
    .filter(F.col("pasajeros") >= 1000)
)
linea_concentracion_pico = guardar(linea_concentracion_pico, "linea_concentracion_pico")
display(linea_concentracion_pico.orderBy(F.desc("pct_en_3_horas_pico")).limit(20))

## 5. Paradas compartidas y huecos de servicio

La parada se relaciona con la línea a través de la variante: `cod_variante` (horarios por parada) corresponde a `variante` (puntos de control), que trae `cod_linea` y el nombre de la línea. Las variantes asociadas a más de una línea se descartan de la correspondencia para no inventar relaciones, y se informa cuántas son.

In [ ]:
control = leer(SILVER, "horarios_puntos_control")

variante_linea = (
    control.filter(F.col("variante").isNotNull() & F.col("cod_linea").isNotNull())
    .groupBy("variante", "cod_linea")
    .agg(F.max("linea").alias("linea"))
)

variantes_ambiguas = (
    variante_linea.groupBy("variante").agg(F.count("*").alias("n_lineas")).filter("n_lineas > 1")
)
variante_linea_ok = variante_linea.join(variantes_ambiguas, "variante", "left_anti")

variantes_horarios = horarios.select("cod_variante").distinct()
resumen_variantes = variantes_horarios.agg(
    F.count("*").alias("variantes_en_horarios"),
    F.sum(
        F.when(
            F.col("cod_variante").isin([r[0] for r in variante_linea_ok.select("variante").collect()]), 1
        ).otherwise(0)
    ).alias("con_linea_unica"),
)
display(resumen_variantes)
print("Variantes asociadas a más de una línea (descartadas):", variantes_ambiguas.count())

In [ ]:
parada_variante = (
    horarios.select("cod_ubic_parada", "cod_variante").distinct()
    .join(variante_linea_ok, F.col("cod_variante") == F.col("variante"), "left")
)

paradas_compartidas = (
    parada_variante.groupBy("cod_ubic_parada")
    .agg(
        F.countDistinct("cod_variante").alias("variantes"),
        F.countDistinct("cod_linea").alias("lineas"),
        F.sort_array(F.collect_set("linea")).alias("nombres_lineas"),
        F.sum(F.when(F.col("cod_linea").isNull(), 1).otherwise(0)).alias("variantes_sin_linea"),
    )
    .withColumnRenamed("cod_ubic_parada", "codigo_parada")
    .withColumn("ranking", F.rank().over(W.orderBy(F.desc("lineas"))))
)
paradas_compartidas = guardar(paradas_compartidas, "paradas_compartidas")
display(paradas_compartidas.orderBy("ranking").limit(20))

paradas_lineas_distribucion = (
    paradas_compartidas.groupBy("lineas").agg(F.count("*").alias("paradas")).orderBy("lineas")
)
paradas_lineas_distribucion = guardar(paradas_lineas_distribucion, "paradas_lineas_distribucion")

### Huecos de servicio

Franja diurna de día hábil: horas 7 a 19 (13 horas). Una hora tiene **headway mayor a 30 min** si hay menos de 2 pasadas. `tiene_hueco` = alguna hora sin servicio o con headway mayor a 30 min.

In [ ]:
todas_paradas = horarios.select(F.col("cod_ubic_parada").alias("codigo_parada")).distinct()

diurno_habil = oferta_parada_hora.filter((F.col("tipo_dia") == "HABIL") & F.col("hora").between(7, 19))

huecos_servicio = (
    todas_paradas.join(
        diurno_habil.groupBy("codigo_parada").agg(
            F.count("*").alias("horas_con_servicio"),
            F.sum((F.col("salidas") < 2).cast("int")).alias("horas_headway_mayor_30"),
            F.max("headway_min").alias("peor_headway_min"),
            F.sum("salidas").alias("salidas_diurnas"),
        ),
        "codigo_parada",
        "left",
    )
    .fillna(0, subset=["horas_con_servicio", "horas_headway_mayor_30", "salidas_diurnas"])
    .withColumn("horas_sin_servicio", F.lit(13) - F.col("horas_con_servicio"))
    .withColumn("tiene_hueco", (F.col("horas_sin_servicio") > 0) | (F.col("horas_headway_mayor_30") > 0))
)
huecos_servicio = guardar(huecos_servicio, "huecos_servicio")
display(huecos_servicio.agg(F.count("*").alias("paradas"), F.sum(F.col("tiene_hueco").cast("int")).alias("con_hueco")))

## 6. Demanda contra oferta por parada

- demanda: pasajeros promedio por día que suben en la parada, en los últimos `meses_demanda` meses completos
- oferta: pasadas programadas en esa parada y hora

`pasajeros_por_salida` (demanda dividida por oferta) indica cuánta gente le toca a cada ómnibus: valores altos sugieren paradas con servicio corto para su demanda. 

`percentil_saturacion` ordena las paradas de 0 a 1 dentro de cada tipo de día y hora.

La columna estado aclara cómo leer cada fila: `CON_AMBOS` (hay demanda y oferta), `SIN_OFERTA_CON_DEMANDA` (suben pasajeros pero no hay pasadas programadas, lo que suele indicar un código de parada que no cruza entre viajes y horarios) y `OFERTA_SIN_DEMANDA` (hay pasadas pero no se registraron pasajeros).

In [ ]:
demanda_parada_hora = (
    con_calendario(
        viajes_silver.filter(fila_valida & F.col("codigo_parada_origen").isNotNull())
        .filter((F.to_date("fecha_evento") >= F.lit(VENTANA_DESDE)) & (F.to_date("fecha_evento") < F.lit(VENTANA_HASTA)))
    )
    .groupBy("codigo_parada_origen", "tipo_dia", "hora")
    .agg(F.sum("cantidad_pasajeros").alias("pasajeros"))
    .join(dias_ventana, "tipo_dia")
    .withColumn("pasajeros_por_dia", F.col("pasajeros") / F.col("dias"))
    .select(F.col("codigo_parada_origen").alias("codigo_parada"), "tipo_dia", "hora", "pasajeros_por_dia")
)

demanda_oferta_parada_hora = (
    demanda_parada_hora.join(
        oferta_parada_hora.select("codigo_parada", "tipo_dia", "hora", "salidas"),
        ["codigo_parada", "tipo_dia", "hora"],
        "full",
    )
    .fillna(0, subset=["pasajeros_por_dia", "salidas"])
    .withColumn("pasajeros_por_salida", F.when(F.col("salidas") > 0, F.col("pasajeros_por_dia") / F.col("salidas")))
    .withColumn(
        "estado",
        F.when((F.col("salidas") == 0) & (F.col("pasajeros_por_dia") > 0), "SIN_OFERTA_CON_DEMANDA")
        .when((F.col("salidas") > 0) & (F.col("pasajeros_por_dia") == 0), "OFERTA_SIN_DEMANDA")
        .otherwise("CON_AMBOS"),
    )
    .withColumn(
        "percentil_saturacion",
        F.percent_rank().over(W.partitionBy("tipo_dia", "hora").orderBy("pasajeros_por_salida")),
    )
)
demanda_oferta_parada_hora = guardar(demanda_oferta_parada_hora, "demanda_oferta_parada_hora")
display(demanda_oferta_parada_hora.groupBy("estado").count())

## 7. Cobertura por zona y hora (mapa)

**Requiere `movilidad_silver.paradas_ubicacion`** (paradas con lat/lon y hexágono H3 nivel 8). Si todavía no existe, este bloque se saltea.

Zona = hexágono H3 de resolución 8. Solo se consideran los hexágonos que tienen al menos una parada, así que el rojo significa **parada sin servicio en esa hora**, no ausencia de red.

`salidas_max_parada` es la mayor cantidad de pasadas por hora en una sola parada del hexágono. Representa la mejor frecuencia a la que accede alguien de la zona, sin contar dos veces la misma línea que pasa por varias paradas. Niveles:
- `SIN_SERVICIO`: 0 pasadas
- `BAJA`: 1 por hora (headway de 60 min)
- `MEDIA`: 2 o 3 por hora (headway de 20 a 30 min)
- `ALTA`: 4 o más por hora (headway de 15 min o menos)

In [ ]:
try:
    paradas_geo = leer(SILVER, "paradas_ubicacion")
    paradas_geo.limit(1).count()
    hay_geo = True
except Exception as e:
    hay_geo = False
    print("Falta movilidad_silver.paradas_ubicacion; se saltea el bloque 7.\nDetalle:", str(e)[:200])

In [ ]:
if hay_geo:
    paradas_hex = (
        paradas_geo.filter(sin_problemas())
        .select("codigo_parada", "h3_8", "lat", "lon")
        .filter(F.col("h3_8").isNotNull())
    )

    hexagonos = (
        paradas_hex.groupBy("h3_8")
        .agg(F.count("*").alias("paradas_en_hex"), F.avg("lat").alias("lat"), F.avg("lon").alias("lon"))
        .withColumn("h3", F.expr("h3_h3tostring(h3_8)"))
    )

    horas = spark.range(24).select(F.col("id").cast("int").alias("hora"))
    tipos = spark.createDataFrame([("HABIL",), ("SABADO",), ("DOMINGO",)], ["tipo_dia"])
    grilla = hexagonos.crossJoin(horas).crossJoin(tipos)

    servicio_hex = (
        oferta_parada_hora.join(paradas_hex, "codigo_parada")
        .groupBy("h3_8", "tipo_dia", "hora")
        .agg(
            F.max("salidas").alias("salidas_max_parada"),
            F.sum("salidas").alias("salidas_total"),
            F.countDistinct("codigo_parada").alias("paradas_con_servicio"),
        )
    )

    cobertura_hex_hora = (
        grilla.join(servicio_hex, ["h3_8", "tipo_dia", "hora"], "left")
        .fillna(0, subset=["salidas_max_parada", "salidas_total", "paradas_con_servicio"])
        .withColumn(
            "headway_min",
            F.when(F.col("salidas_max_parada") > 0, F.lit(60.0) / F.col("salidas_max_parada")),
        )
        .withColumn(
            "nivel",
            F.when(F.col("salidas_max_parada") == 0, "SIN_SERVICIO")
            .when(F.col("salidas_max_parada") < 2, "BAJA")
            .when(F.col("salidas_max_parada") < 4, "MEDIA")
            .otherwise("ALTA"),
        )
        .withColumn("pct_paradas_con_servicio", 100 * F.col("paradas_con_servicio") / F.col("paradas_en_hex"))
    )
    cobertura_hex_hora = guardar(cobertura_hex_hora, "cobertura_hex_hora")
    display(cobertura_hex_hora.groupBy("tipo_dia", "nivel").count().orderBy("tipo_dia", "nivel"))

### Desiertos nocturnos

Hexágonos con servicio durante el día hábil (horas 7 a 19) y **sin ninguna pasada en la madrugada** (horas 0 a 5).

In [ ]:
if hay_geo:
    desiertos_nocturnos = (
        cobertura_hex_hora.filter(F.col("tipo_dia") == "HABIL")
        .groupBy("h3", "lat", "lon", "paradas_en_hex")
        .agg(
            F.sum(F.when(F.col("hora").between(7, 19) & (F.col("salidas_max_parada") > 0), 1).otherwise(0)).alias("horas_diurnas_con_servicio"),
            F.sum(F.when(F.col("hora").between(0, 5) & (F.col("salidas_max_parada") > 0), 1).otherwise(0)).alias("horas_madrugada_con_servicio"),
        )
        .withColumn(
            "desierto_nocturno",
            (F.col("horas_diurnas_con_servicio") > 0) & (F.col("horas_madrugada_con_servicio") == 0),
        )
    )
    desiertos_nocturnos = guardar(desiertos_nocturnos, "desiertos_nocturnos")
    display(desiertos_nocturnos.agg(F.count("*").alias("hexagonos"), F.sum(F.col("desierto_nocturno").cast("int")).alias("desiertos")))

### Demanda contra oferta por zona

In [ ]:
if hay_geo:
    demanda_oferta_hex_hora = (
        demanda_oferta_parada_hora.join(paradas_hex, "codigo_parada")
        .groupBy("h3_8", "tipo_dia", "hora")
        .agg(
            F.sum("pasajeros_por_dia").alias("pasajeros_por_dia"),
            F.sum("salidas").alias("salidas_total"),
        )
        .withColumn("pasajeros_por_salida", F.when(F.col("salidas_total") > 0, F.col("pasajeros_por_dia") / F.col("salidas_total")))
        .join(hexagonos.select("h3_8", "h3", "lat", "lon"), "h3_8")
    )
    demanda_oferta_hex_hora = guardar(demanda_oferta_hex_hora, "demanda_oferta_hex_hora")
    display(demanda_oferta_hex_hora.orderBy(F.desc("pasajeros_por_salida")).limit(20))

## 8. Calidad de los datos de viajes (opcional, solo para constatar)

¿Cuántas filas de `viajes_stm` tienen cada marca de validación y cuántas quedan fuera del gold? Una fila puede tener varias marcas, por eso las columnas no suman el total.

Recorrer 1.800 millones de filas solo para contar marcas es lento, así que por defecto se usa una **muestra aleatoria** (`calidad_muestra`, 1 % por defecto). Los porcentajes son una estimación. 

Con `calidad_muestra = 1` el conteo es exacto.

Nada del gold depende de esta tabla.

In [ ]:
base_calidad = viajes_silver if CALIDAD_MUESTRA >= 1 else viajes_silver.sample(False, CALIDAD_MUESTRA, seed=42)

calidad_viajes = base_calidad.agg(
    F.count("*").alias("filas_analizadas"),
    F.sum((~fila_valida).cast("int")).alias("excluidas_del_gold"),
    *[
        F.sum(F.coalesce(F.array_contains("problemas_validacion", r), F.lit(False)).cast("int")).alias(r)
        for r in REGLAS_SILVER
    ],
)
columnas_pct = [
    (100 * F.col(c) / F.col("filas_analizadas")).alias(f"pct_{c}")
    for c in ["excluidas_del_gold"] + REGLAS_SILVER
]
calidad_viajes = calidad_viajes.select("*", *columnas_pct).withColumn("fraccion_muestra", F.lit(CALIDAD_MUESTRA))
calidad_viajes = guardar(calidad_viajes, "calidad_viajes")
display(calidad_viajes)

In [ ]:
## Comprobación de tablas publicadas

In [ ]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{GOLD}"))